# FF49 — Evidential probes and uncertainty-aware routing

Dataset: **`ken_french_49`**, **49 assets**. This transfers an existing KF12 experiment;
it does not assume KF12 outcomes or tune new methods. FF49 is the primary high-dimensional comparison. Its complete-case coverage begins in July 1969; the verified archive has 7,648 training and 1,256 validation windows.

The saved `configs/cde_ff49.yaml` protocol remains unchanged: requested sample **1958–2023**,
training through **1999-12-31**, validation **2000–2004**, lookback **60**, horizon **1**, seed **42**.
Arithmetic decimal returns minus daily French RF; complete-case row exclusion without imputation;
seven-day calendar-gap segmentation and train-only standardization. **No test evaluation.**

**Prerequisite:** completed, consistent seed-42 CDE at `/kaggle/working/ff49_cde_clean`
.
Both references also need their saved validation metrics.
Finish those runs in their own notebooks; this notebook never trains a missing reference.

Work through individual cells. Extraction/training/evaluation and the final upload are manual steps;
**Run All can be expensive and uploads artifacts**. Outputs are isolated at `/kaggle/working/ff49_evidential_router` and
`high_dimensional/ff49_evidential_router` in the central Kaggle dataset. No experiment results are populated here.

## Setup

Enable Internet and a GPU in Kaggle settings. This is the established KF12 GitHub sync cell;
`BRANCH` selects the checkout containing these notebooks. Use `main` after merging.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/high-dimensional-kaggle"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev,cde]'

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Lightweight regression tests

In [ ]:
!pytest -q tests/semantic tests/unit/test_multiseed.py tests/unit/test_evidential_regimes.py \
    tests/unit/test_routing.py tests/unit/test_spd_features.py tests/unit/test_topology.py \
    tests/unit/test_precomputed_conditioning.py tests/unit/test_highdim_research.py
assert _exit_code == 0, "Regression tests failed."

## Restore prerequisites and this experiment

Only the listed reference runs and this notebook's own namespace are restored. Local folders
are retained. Missing references fail the next check; there is no KF12 fallback.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import torch
import kagglehub
from diffusion_portfolio.utils.research_checks import (
    check_checkpoint_pair, require_completed_run, source_identity,
)

HANDLE = "amirentezari/diffusion-portfolio"
EXPERIMENT = "high_dimensional/ff49_evidential_router"
ROOT = Path("/kaggle/working/ff49_evidential_router")
CDE_DIR = Path("/kaggle/working/ff49_cde_clean")
REFERENCES = {Path(path): config for path, config in {'/kaggle/working/ff49_cde_clean': 'configs/cde_ff49.yaml'}.items()}

In [ ]:
artifact_path = Path(kagglehub.dataset_download(HANDLE))
restores = {"high_dimensional/" + path.name: path for path in REFERENCES}
restores[EXPERIMENT] = ROOT
for relative, destination in restores.items():
    source = artifact_path / relative
    if destination.exists():
        print("Already available:", destination)
    elif source.is_dir():
        if destination in REFERENCES:
            require_completed_run(source, REFERENCES[destination], assets=49, validation=True)
        shutil.copytree(source, destination)
        print("Restored:", destination)
    else:
        print("No backup:", relative)

In [ ]:
for path, config in REFERENCES.items():
    require_completed_run(path, config, assets=49, validation=True)
cfg = require_completed_run(CDE_DIR, "configs/cde_ff49.yaml", assets=49)
SOURCE = source_identity(CDE_DIR)
print("Prerequisites complete:", cfg.data.dataset, "assets:", 49, "seed:", cfg.seed)

## Dimensions and regime-target smoke check

Shared data preparation preserves calendar gaps and saved normalization. This diagnostic uses
the saved panel and the existing regime functions, without CDE inference; thresholds are fitted
on training scores only. The CLI probes retain their historical **train-only scaler refit**,
which must agree with the saved CDE scaler before proceeding.

In [ ]:
from diffusion_portfolio.data.preparation import prepare_return_data
from diffusion_portfolio.evaluation.regimes import (
    cross_sectional_rms, trailing_cross_sectional_rms, fit_regime_thresholds, assign_regime_labels,
)

datasets, scaler = prepare_return_data(cfg, standardizer_path=CDE_DIR / "standardizer.npz")
refitted, refit_scaler = prepare_return_data(cfg)
assert np.array_equal(refit_scaler.mean, scaler.mean) and np.array_equal(refit_scaler.std, scaler.std), "Source data changed since the CDE scaler was saved."
for target in ("next_day", "current"):
    train_raw = datasets.train.raw_windows
    val_raw = datasets.val.raw_windows
    train_scores = cross_sectional_rms(train_raw.target[:, 0]) if target == "next_day" else trailing_cross_sectional_rms(train_raw.history, recent_days=20)
    val_scores = cross_sectional_rms(val_raw.target[:, 0]) if target == "next_day" else trailing_cross_sectional_rms(val_raw.history, recent_days=20)
    thresholds = fit_regime_thresholds(train_scores, quantiles=(.333333, .666667))
    labels = assign_regime_labels(val_scores, thresholds=thresholds)
    print(target, "train-only thresholds:", thresholds, "validation class counts:", np.bincount(labels, minlength=3))
assert len(scaler.columns) == 49 and cfg.model.condition_dim == 128
validation_dates = datasets.val.raw_windows.target_dates.values.copy()
del datasets, refitted, train_raw, val_raw  # Free dense windows before embedding extraction.

## Frozen-CDE regime probes

Each command trains the evidential head and equal-capacity softmax control for the existing
**50 epochs**, selects each head by validation NLL, and writes validation-only diagnostics.
Next-day scores use the target cross-sectional RMS; current scores use trailing 20-day RMS.
**These CLIs are single-shot, not resumable.** Do not pass `--resume` or invent pilot flags.
A restored completed step can be inspected below. A partial existing directory is not overwritten;
decide explicitly on a fresh output directory if a single-shot step was interrupted.

### Next Day probe

In [ ]:
assert not (ROOT / "next_day").exists(), "Single-shot probe: inspect the restored run or choose a fresh directory explicitly."

!PYTHONUNBUFFERED=1 python -u scripts/train_evidential_regime.py \
    --cde-run-dir /kaggle/working/ff49_cde_clean \
    --config configs/evidential.yaml \
    --output-dir /kaggle/working/ff49_evidential_router/next_day \
    --regime-target next_day \
    --device cuda
assert _exit_code == 0, "probe-next_day failed."

In [ ]:
import matplotlib.pyplot as plt

run = ROOT / "next_day"
metrics = json.loads((run / "validation_metrics.json").read_text())
assert metrics["source"] == SOURCE
display(pd.DataFrame({head: metrics[head] for head in ("evidential", "softmax")}))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, head in zip(axes, ("evidential", "softmax")):
    history = pd.read_csv(run / f"{head}_history.csv")
    history.plot(x="epoch", y=["train_loss", "val_nll"], ax=ax)
    ax.set(title=f"FF49 next_day: {head}", xlabel="Epoch", ylabel="Training objective / validation NLL")
    ax.grid(True, alpha=.3)
fig.tight_layout()
plt.show()
with np.load(run / "validation_predictions.npz") as predictions:
    assert np.array_equal(predictions["dates"], validation_dates)
    display(pd.crosstab(predictions["labels"], predictions["evidential_probabilities"].argmax(1), rownames=["True"], colnames=["Predicted"]))
    display(pd.DataFrame({"Regime": predictions["labels"], "Vacuity": predictions["evidential_vacuity"]}).groupby("Regime").agg(["count", "mean"]))

### Current probe

In [ ]:
assert not (ROOT / "current").exists(), "Single-shot probe: inspect the restored run or choose a fresh directory explicitly."

!PYTHONUNBUFFERED=1 python -u scripts/train_evidential_regime.py \
    --cde-run-dir /kaggle/working/ff49_cde_clean \
    --config configs/evidential.yaml \
    --output-dir /kaggle/working/ff49_evidential_router/current \
    --regime-target current \
    --device cuda
assert _exit_code == 0, "probe-current failed."

In [ ]:
import matplotlib.pyplot as plt

run = ROOT / "current"
metrics = json.loads((run / "validation_metrics.json").read_text())
assert metrics["source"] == SOURCE
display(pd.DataFrame({head: metrics[head] for head in ("evidential", "softmax")}))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, head in zip(axes, ("evidential", "softmax")):
    history = pd.read_csv(run / f"{head}_history.csv")
    history.plot(x="epoch", y=["train_loss", "val_nll"], ax=ax)
    ax.set(title=f"FF49 current: {head}", xlabel="Epoch", ylabel="Training objective / validation NLL")
    ax.grid(True, alpha=.3)
fig.tight_layout()
plt.show()
with np.load(run / "validation_predictions.npz") as predictions:
    assert np.array_equal(predictions["dates"], validation_dates)
    display(pd.crosstab(predictions["labels"], predictions["evidential_probabilities"].argmax(1), rownames=["True"], colnames=["Predicted"]))
    display(pd.DataFrame({"Regime": predictions["labels"], "Vacuity": predictions["evidential_vacuity"]}).groupby("Regime").agg(["count", "mean"]))

## Uncertainty-aware routing (current-regime heads only)

CDE/denoiser and regime heads stay frozen. Three zero-initialized residual experts are matched
between softmax and evidential routing; only experts train for the original 30 epochs.
Tau is the **training median vacuity**, gamma stays **10**. The CLI evaluates both best-validation
routers and emits the CDE/softmax/evidential comparison. This step is also single-shot.
Router validation retains its historical interleaved corruption stream; its CDE control loss
can differ from the saved diffusion checkpoint's separate-stream validation loss.

In [ ]:
probe = ROOT / "current"
for filename in ("evidential_best.pt", "softmax_best.pt", "regime_thresholds.json", "validation_metrics.json"):
    assert (probe / filename).is_file(), f"Complete the current-regime probe first: {probe / filename}"
assert json.loads((probe / "regime_thresholds.json").read_text())["regime_target"] == "current"
assert json.loads((probe / "validation_metrics.json").read_text())["source"] == SOURCE
for filename in ("evidential_best.pt", "softmax_best.pt"):
    head = torch.load(probe / filename, map_location="cpu", weights_only=False)
    assert head["input_dim"] == cfg.model.condition_dim and head["n_classes"] == 3

In [ ]:
assert not (ROOT / "router").exists(), "Single-shot router: do not overwrite restored or partial results."

!PYTHONUNBUFFERED=1 python -u scripts/train_router.py \
    --cde-run-dir /kaggle/working/ff49_cde_clean \
    --probe-run-dir /kaggle/working/ff49_evidential_router/current \
    --config configs/router.yaml \
    --output-dir /kaggle/working/ff49_evidential_router/router \
    --device cuda
assert _exit_code == 0, "router failed."

In [ ]:
import matplotlib.pyplot as plt

comparison = json.loads((ROOT / "router/comparison.json").read_text())
assert comparison["source"] == SOURCE
display(pd.DataFrame({name: comparison[name] for name in ("CDE", "Softmax MoE", "Evidential MoE")}).T)
print("Control corruption stream:", comparison.get("baseline_validation_stream", "router_legacy"))
print("Saved CDE checkpoint loss:", comparison.get("checkpoint_validation_loss", comparison["CDE"]["validation_loss"]))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, variant in zip(axes, ("softmax_moe", "evidential_moe")):
    history = pd.read_csv(ROOT / "router" / variant / "history.csv")
    history.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax)
    ax.set(title=variant, xlabel="Epoch", ylabel="Router diffusion loss")
    ax.grid(True, alpha=.3)
    metrics = json.loads((ROOT / "router" / variant / "validation_probabilistic_metrics.json").read_text())
    assert metrics["split"] == "validation"
    display(pd.DataFrame(metrics["calibration"]).assign(Variant=variant))
    display(pd.Series(metrics["routing"], name=variant).to_frame())
fig.tight_layout()
plt.show()

## Manual backup

Click only when ready to upload. The latest central dataset is copied in full, then only this
notebook's own namespace is replaced. **Serialize uploads across Kaggle sessions**; there is no
cross-session lock or automatic merge. A failed/empty download stops upload. Prerequisite seed-42
directories are never replaced here. Resumable diffusion checkpoints can be backed up mid-run;
probe/router steps need their completed per-step artifacts because those CLIs are single-shot.

In [ ]:
assert ROOT.is_dir(), "No probe/router artifacts to back up."
for target in ("next_day", "current"):
    probe = ROOT / target
    if probe.exists():
        for filename in ("evidential_best.pt", "softmax_best.pt", "regime_thresholds.json", "validation_metrics.json", "validation_predictions.npz", "config.yaml"):
            assert (probe / filename).is_file(), f"Incomplete single-shot probe: {probe}"
        assert json.loads((probe / "validation_metrics.json").read_text())["source"] == SOURCE
if (ROOT / "router").exists():
    assert (ROOT / "router/comparison.json").is_file(), "Router evaluation has not completed."
    assert json.loads((ROOT / "router/comparison.json").read_text())["source"] == SOURCE
assert any(ROOT.rglob("evidential_best.pt")), "No completed probe checkpoint."

for last in ROOT.rglob("last.pt"):
    check_checkpoint_pair(last.parent)
artifact_path = Path(kagglehub.dataset_download(HANDLE, force_download=True))
if not any(p.is_file() and p.name != ".complete" for p in artifact_path.rglob("*")):
    raise RuntimeError("Empty central artifact dataset; refusing to replace it.")
staging = ROOT.parent / (ROOT.name + "_artifact_sync")
if staging.exists():
    shutil.rmtree(staging)
shutil.copytree(artifact_path, staging)
destination = staging / EXPERIMENT
if destination.exists():
    shutil.rmtree(destination)
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(ROOT, destination)
kagglehub.dataset_upload(HANDLE, str(staging), ignore_patterns=[".complete"],
                         version_notes=f"Available research artifacts: {EXPERIMENT}")
print("Backed up:", EXPERIMENT)